In [12]:
import pandas as pd

# Zillow Home Value Index Data Cleaning

This notebook documents the cleaning process for the Zillow Home Value Index (ZHVI) dataset from 2015 to 2024.

The goal is to remove unnecessary columns, calculate annual average home values for each state, and prepare the data for integration with our other datasets.

In [13]:
df = pd.read_csv("zillow_home_values_raw.csv")

df.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,2000-03-31,2000-04-30,2000-05-31,2000-06-30,2000-07-31,...,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31,2026-09-30,2026-10-31
0,9,0,California,state,NaN,188543.392043,190386.342723,192528.495264,194839.871633,197280.850172,...,760934.105939,761286.927566,760594.750978,759258.575152,757988.155962,756954.088055,757229.551056,758513.537603,759862.550007,760611.575627
1,54,1,Texas,state,NaN,113285.472437,113433.544105,113529.437639,113636.957635,113707.614908,...,298397.994321,298377.968321,298094.089081,297510.319909,296575.423254,295615.038628,294952.308077,294704.272839,294558.377525,294310.928135
2,14,2,Florida,state,NaN,109009.872510,109588.932187,110213.385648,110850.529370,111490.702969,...,375829.570651,376013.296112,376109.690628,375729.795238,375028.208723,374316.060041,374105.629041,374298.134196,374661.828008,375056.002008
3,43,3,New York,state,NaN,152169.159376,153283.276239,154435.988931,155682.390132,156802.900386,...,503352.784688,505733.275091,508382.825759,510706.617474,512694.472240,514703.258430,517283.549539,520133.683855,522138.484166,523216.805664
4,47,4,Pennsylvania,state,NaN,100022.969834,100430.981469,100850.111579,101280.863944,101738.296388,...,281629.686588,282336.054246,282873.713247,283196.157354,283441.296150,283932.674138,284538.721657,285291.991794,285687.881922,285896.996915


In [14]:
date_cols = [
    col for col in df.columns
    if col.startswith(tuple(str(y) for y in range(2015, 2025)))
]

df = df[["RegionName"] + date_cols]

df.head()

,RegionName,2015-01-31,2015-02-28,2015-03-31,2015-04-30,2015-05-31,2015-06-30,2015-07-31,2015-08-31,2015-09-30,...,2024-03-31,2024-04-30,2024-05-31,2024-06-30,2024-07-31,2024-08-31,2024-09-30,2024-10-31,2024-11-30,2024-12-31
0,California,400333.650595,402749.069694,404726.057397,406639.553381,408803.149778,411020.386936,412767.562050,413938.489904,414940.030482,...,753856.730919,757703.600830,762176.008282,764639.284268,766628.500280,768391.266084,770679.723127,772265.039347,774032.386169,775820.299020
1,Texas,161307.642467,162566.452867,163797.111604,164948.762235,166050.614898,167148.163032,168289.319407,169479.842100,170693.771313,...,307058.699217,307930.701331,308267.991191,308032.237722,307461.660369,307039.179288,306775.582687,306513.199831,306137.979200,305861.488036
2,Florida,175420.156609,176666.052958,177766.114664,178870.951721,180122.389623,181405.535169,182868.558773,184417.416627,186102.284115,...,402450.169920,403139.832209,403266.095362,402469.109889,401397.812044,400367.546622,399499.394462,398537.274747,397318.720868,396208.172736
3,New York,270030.395868,270275.144026,270523.660326,270974.683931,271784.602763,272672.303340,273777.379720,275102.949187,276856.045076,...,457758.778961,461396.893495,464740.859953,467429.692076,469943.586607,473019.563976,475863.178868,478265.380246,480084.433421,481827.240627
4,Pennsylvania,159334.460183,159387.036010,159580.437119,160005.115113,160604.805314,161139.778162,161776.699525,162508.451759,163224.892176,...,265609.137779,267305.991819,268817.109607,269716.645737,270059.716788,270313.769865,270706.719883,271461.811860,272362.881688,273569.629061


This removes the extra metadata columns and keeps only the months from 2015 through 2024.

In [15]:
df = df.melt(
    id_vars="RegionName",
    var_name="Date",
    value_name="Home_Value"
)

df.head()

,RegionName,Date,Home_Value
0,California,2015-01-31,400333.650595
1,Texas,2015-01-31,161307.642467
2,Florida,2015-01-31,175420.156609
3,New York,2015-01-31,270030.395868
4,Pennsylvania,2015-01-31,159334.460183


## Calculating Annual Averages

Since the original dataset contains monthly home values, I grouped the data by state and year and calculated the average home value for each year. This makes the Zillow data consistent with the yearly format used in our other datasets.

In [16]:
df["Year"] = pd.to_datetime(df["Date"]).dt.year

df = df.groupby(
    ["RegionName", "Year"], as_index=False
)["Home_Value"].mean()

df.head()

,RegionName,Year,Home_Value
0,Alabama,2015,137276.872970
1,Alabama,2016,142223.347907
2,Alabama,2017,146347.248841
3,Alabama,2018,152181.902310
4,Alabama,2019,160637.393281


In [17]:
df = df.rename(columns={
    "RegionName": "State",
    "Home_Value": "Avg_Home_Values"
})

df = df[df["State"] != "District of Columbia"]

df.head()

,State,Year,Avg_Home_Values
0,Alabama,2015,137276.872970
1,Alabama,2016,142223.347907
2,Alabama,2017,146347.248841
3,Alabama,2018,152181.902310
4,Alabama,2019,160637.393281


## Data Quality Checks

I checked the cleaned dataset for missing values, duplicate records, and the total number of states. The final dataset should contain 500 records, representing all 50 states from 2015 to 2024.

In [18]:
print("Rows:", len(df))
print("States:", df["State"].nunique())
print("Missing values:")
print(df.isnull().sum())

Rows: 500
States: 50
Missing values:
State              0
Year               0
Avg_Home_Values    0
dtype: int64


In [19]:
print("Duplicates:", df.duplicated(subset=["State", "Year"]).sum())

Duplicates: 0
